## Import Library

In [ ]:
"""GradCAM++ exploration and validation on NIH ChestX-ray14 samples.
 
Loads the trained DenseNet-121 and per-class thresholds, runs GradCAM++
on a sample of test images, and saves annotated heatmaps per condition
for visual inspection.
 
Output: /kaggle/working/gradcam_samples/<condition>/<image_id>.png
 
Reference: Chattopadhay et al. (2018), GradCAM++, WACV 2018.
"""
 
import json
import numpy as np
from pathlib import Path
from PIL import Image
 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import timm
 
from dataset_utils import (
    ALL_CONDITIONS,
    build_image_index,
    load_multilabel_csv,
    patient_level_split,
    ChestXrayDataset,
    get_eval_transform,
    IMAGE_SIZE,
)

## Config

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = Path("/kaggle/working/multilabel_model.pt")
THRESHOLD_PATH = Path("/kaggle/working/multilabel_thresholds.json")
OUTPUT_DIR = Path("/kaggle/working/gradcam_samples")
N_SAMPLES = 5      # images per condition to visualize
 
# 7-zone boundaries (x1, y1, x2, y2) normalized — Felson (1973)
CHEST_REGIONS: dict[str, tuple] = {
    "RUZ": (0.00, 0.00, 0.50, 0.33),
    "LUZ": (0.50, 0.00, 1.00, 0.33),
    "RMZ": (0.00, 0.33, 0.42, 0.63),
    "LMZ": (0.58, 0.33, 1.00, 0.63),
    "CAR": (0.33, 0.30, 0.67, 0.73),
    "RLZ": (0.00, 0.63, 0.50, 1.00),
    "LLZ": (0.50, 0.63, 1.00, 1.00),
}

## Helper Fucntion

In [ ]:
def load_model() -> nn.Module:
    """Load trained DenseNet-121 in eval mode."""
    model = timm.create_model(
        "densenet121", pretrained=False, num_classes=14, drop_rate=0.2
    )
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model
 
 
def compute_zone_stats(heatmap: np.ndarray) -> dict[str, float]:
    """Compute mean activation per anatomical zone from GradCAM heatmap."""
    h = w = IMAGE_SIZE
    stats = {}
    for zone, (x1, y1, x2, y2) in CHEST_REGIONS.items():
        region = heatmap[int(y1*h):int(y2*h), int(x1*w):int(x2*w)]
        stats[zone] = round(float(region.mean()), 4) if region.size > 0 else 0.0
    return stats
 
 
def save_heatmap(
    heatmap: np.ndarray,
    rgb_array: np.ndarray,
    save_path: Path,
    zone_stats: dict,
) -> None:
    """Save GradCAM overlay image with zone stats annotation."""
    cam_image = show_cam_on_image(rgb_array, heatmap, use_rgb=True)
    Image.fromarray(cam_image).save(save_path)

In [ ]:
def run_gradcam_exploration(
    model: nn.Module,
    df_test,
    image_index: dict,
    thresholds: dict[str, float],
) -> None:
    """Run GradCAM++ on N_SAMPLES test images per condition and save outputs."""
    target_layer = model.features.denseblock4
    transform    = get_eval_transform()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
    with open(THRESHOLD_PATH) as f:
        thresholds = json.load(f)
 
    with GradCAMPlusPlus(model=model, target_layers=[target_layer]) as cam:
        for cond_idx, condition in enumerate(ALL_CONDITIONS):
            cond_dir = OUTPUT_DIR / condition
            cond_dir.mkdir(exist_ok=True)
 
            # Sample test images positive for this condition
            positive_rows = df_test[df_test[condition] == 1].head(N_SAMPLES)
            if positive_rows.empty:
                print(f"No positive samples for {condition} in test set")
                continue
 
            for _, row in positive_rows.iterrows():
                img_path = image_index.get(row["Image Index"])
                if img_path is None:
                    continue
 
                with Image.open(img_path) as img:
                    img_rgb = img.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
                    rgb_arr = np.array(img_rgb, dtype=np.float32) / 255.0
                    tensor = transform(img).unsqueeze(0).to(DEVICE)
 
                heatmap = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(cond_idx)])[0]
                zone_stats = compute_zone_stats(heatmap)
                save_path = cond_dir / f"{row['Image Index']}.png"
 
                save_heatmap(heatmap, rgb_arr, save_path, zone_stats)
 
                dominant = max(zone_stats, key=zone_stats.get)
                print(f"  {condition} | {row['Image Index']} | dominant: {dominant} ({zone_stats[dominant]:.3f})")
 
    print(f"\nSamples saved to {OUTPUT_DIR}")

## Main Run

In [ ]:
if __name__ == "__main__":
    IMAGE_INDEX = build_image_index()
    df = load_multilabel_csv(IMAGE_INDEX)
    _, _, df_test = patient_level_split(df)
 
    with open(THRESHOLD_PATH) as f:
        thresholds = json.load(f)
 
    model = load_model()
    run_gradcam_exploration(model, df_test, IMAGE_INDEX, thresholds)